# 🎙️ VoxCPM2 Khmer Voice LoRA Fine-Tuning Pipeline
### ការ Train (Fine-Tuning LoRA) លើ Google Colab GPU សម្រាប់សំឡេងខ្មែរដូចមនុស្សពិត 90% - 95%

Notebook នេះរៀបចំឡើងជាពិសេសសម្រាប់ Train **LoRA (Low-Rank Adaptation)** លើ **OpenBMB VoxCPM2** ៖
- ⚡ **Hardware:** ដំណើរការលើ Google Colab GPU (T4 ឥតគិតថ្លៃ ឬ A100 / V100)
- ⏱️ **រយៈពេល Train:** ប្រហែល 30 នាទី ទៅ 1 ម៉ោងកន្លះ (1,000 steps)
- 📦 **ទំហំ File ចុងក្រោយ:** `lora_weights.safetensors` (~50 MB ទៅ 100 MB) ងាយស្រួលទាញយកមកដាក់ក្នុង AI Video Editor នៅលើ Mac mini

## ជំហានទី ១: ពិនិត្យមើល GPU (Check NVIDIA GPU)
សូមប្រាកដថាអ្នកបានបើក GPU Runtime (ចុច **Runtime** > **Change runtime type** > ជ្រើសរើស **T4 GPU** ឬ **A100 GPU**)

In [ ]:
!nvidia-smi

## ជំហានទី ២: Clone OpenBMB/VoxCPM និងតម្លើង Dependencies

In [ ]:
# 1. Clone repository ផ្លូវការ
!rm -rf VoxCPM
!git clone https://github.com/OpenBMB/VoxCPM.git
%cd VoxCPM

# 2. Install package និង dependencies ចាំបាច់
!pip install -q -e .
!pip install -q accelerate datasets soundfile librosa argbind pyyaml safetensors
print("✅ VoxCPM2 environment setup complete!")

## ជំហានទី ៣: បញ្ចូល Dataset (`voxcpm_dataset.zip`)
អ្នកអាច Upload file `voxcpm_dataset.zip` ដែលបង្កើតដោយ `prepare_dataset.py` ពីកុំព្យូទ័ររបស់អ្នក ឬភ្ជាប់ជាមួយ Google Drive។

In [ ]:
import os, zipfile
from google.colab import files

# បង្កើតថត dataset
!mkdir -p /content/VoxCPM/dataset

print("សូមជ្រើសរើស file `voxcpm_dataset.zip` ដើម្បី Upload:")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.zip'):
        print(f"📦 ពន្លា file {fn}...")
        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall('/content/VoxCPM/dataset')
        print("✅ ពន្លា Dataset រួចរាល់!")

# ពិនិត្យមើលទិន្នន័យ
!ls -la /content/VoxCPM/dataset
!head -n 3 /content/VoxCPM/dataset/train.jsonl

## ជំហានទី ៤: ផ្ទៀងផ្ទាត់ Dataset (Pre-Flight Validation)
VoxCPM មាន Tool សម្រាប់ពិនិត្យមើលក្រែងមាន Error ក្នុង Dataset មុនពេល Train

In [ ]:
!voxcpm validate --manifest /content/VoxCPM/dataset/train.jsonl

## ជំហានទី ៥: បង្កើត Configuration សម្រាប់ Train LoRA (`voxcpm_khmer_lora.yaml`)

In [ ]:
config_yaml = """
pretrained_path: openbmb/VoxCPM2
train_manifest: /content/VoxCPM/dataset/train.jsonl
sample_rate: 16000
batch_size: 1
grad_accum_steps: 16
learning_rate: 0.0001
weight_decay: 0.01
warmup_steps: 50
num_iters: 1000
save_path: /content/checkpoints/khmer_lora
max_batch_tokens: 8192

lora:
  enable_lm: true
  enable_dit: true
  enable_proj: false
  r: 32
  alpha: 16
  dropout: 0.0
"""

with open("/content/VoxCPM/voxcpm_khmer_lora.yaml", "w", encoding="utf-8") as f:
    f.write(config_yaml.strip())

print("✅ Configuration file created at: /content/VoxCPM/voxcpm_khmer_lora.yaml")

## ជំហានទី ៦: ចាប់ផ្តើមដំណើរការ Train LoRA (Start Fine-Tuning) 🔥
ដំណើរការនេះនឹងរត់ចំនួន 1,000 steps ដោយរក្សាទុក Checkpoint ក្នុង `/content/checkpoints/khmer_lora/`

In [ ]:
!mkdir -p /content/checkpoints/khmer_lora
!python scripts/train_voxcpm_finetune.py --config_path voxcpm_khmer_lora.yaml

## ជំហានទី ៧: សាកល្បងសំឡេងក្រោយពេល Train រួច (Test LoRA Voice Synthesis)
ស្តាប់សំឡេងដែលបាន Fine-tune ភ្លាមៗនៅក្នុង Colab

In [ ]:
import glob, os
import soundfile as sf
from IPython.display import Audio, display
from voxcpm import VoxCPM

# ស្វែងរក checkpoint ចុងក្រោយ
ckpt_candidates = glob.glob("/content/checkpoints/khmer_lora/**/lora_weights.*", recursive=True)
print("Found LoRA weights:", ckpt_candidates)

if ckpt_candidates:
    lora_path = ckpt_candidates[0]
    print(f"🎯 Loading trained LoRA adapter: {lora_path}")
    
    model = VoxCPM.from_pretrained(
        "openbmb/VoxCPM2",
        lora_weights_path=lora_path,
        load_denoiser=False,
        device="cuda"
    )
    
    test_sentence = "សួស្តីបងប្អូនទាំងអស់គ្នា នេះជាសំឡេងខ្មែរដែលបាន Train ជាមួយ LoRA រួចរាល់។"
    wav = model.generate(text=test_sentence, inference_timesteps=20)
    
    sf.write("/content/test_khmer_lora.wav", wav, 48000)
    print("🔊 ស្តាប់សំឡេងសាកល្បង:")
    display(Audio("/content/test_khmer_lora.wav", rate=48000))
else:
    print("⚠️ រកមិនឃើញ checkpoint ទេ សូមពិនិត្យជំហាន Train")

## ជំហានទី ៨: ទាញយក File LoRA មកប្រើក្នុង AI Video Editor លើ Mac mini
File នេះមានទំហំប្រហែល ~50MB ទៅ 100MB ។ នៅពេលទាញយករួច សូម copy ទៅកាន់ folder:
`AI VIDEO EDITOR/backend/python/models/lora/khmer_voice.safetensors`

In [ ]:
import glob
from google.colab import files

ckpt_candidates = glob.glob("/content/checkpoints/khmer_lora/**/lora_weights.*", recursive=True)
if ckpt_candidates:
    target_file = ckpt_candidates[0]
    print(f"⬇️ កំពុងទាញយក file: {target_file}...")
    files.download(target_file)
    print("🎉 ទាញយករួចរាល់!")
else:
    print("⚠️ មិនមាន file សម្រាប់ទាញយកទេ")